# colony32

> Phase 18, a user-approved deviation (`specs/multiagent/phase18/`): guards at 32 agents. The frozen go/no-go rule stopped the 32-64 agent stage because the guards' advantage was smaller at 16 agents than at 8, but that decline was not significant (two-sided exact p = 0.17), so the user approved one more size. The plain board and the board with guards at N = 32, clean and with three kinds of faulty members (independent liars, mistaken members, colluding liars), the Stage 1 seeds; `COLONY32_PLAN` (one primary test, frozen before the run), `stage32Grid`, `stage32Summaries` and `writeStage32`.

In [ ]:
#| default_exp colony32

In [ ]:
#| hide
from nbdev.showdoc import *

## Grid and plan

In [ ]:
#| export
import numpy as np

from explRL.colony_grid import COLONY_SEEDS, colonyCell
from explRL.colony_report import NAN, bootSeeds, cellVals, pairs, signFlipExact, trendSlopes

STAGE32_ARMS, STAGE32_FAULTS, N32 = ('plain', 'colonyGuards'), ('clean', 'independent', 'error', 'collude'), 32


def stage32Grid(seeds=COLONY_SEEDS):
    "Study colony32: plain and colonyGuards at N 32, clean and with each faulty-member kind (8 cells per seed)."
    return [colonyCell('colony32', a, N32, s, f) for a in STAGE32_ARMS for f in STAGE32_FAULTS for s in seeds]


COLONY32_PLAN = dict(
    version=1, frozen=True, deviation=dict(
        date='2026-10-20', approvedBy='user', of='COLONY_PLAN go/no-go (stop: the guard advantage fell from 0.26 at N 8 to 0.20 '
        'at N 16)', why='the decline was not significant (two-sided exact sign-flip p = 0.17, 95% interval [-0.129, +0.003]); '
        'one more size tests whether guards hold in larger teams', instead='COLONY2_PLAN (N 32 and 64) is not run'),
    cells='stage32Grid: plain and colonyGuards x clean, independent, error (mistaken), collude x N 32 x seeds 1800-1807',
    cap=200.0, capNote='raised from $150 by the user for this run (about 67,000 calls, roughly $35-45)',
    primary=dict(H18_7='colonyGuards - plain, honestPooled, independent liars, N 32, 8 seeds paired: one-sided exact sign-flip > 0',
                 alpha=0.05),
    secondary=dict(
        H18_8='colonyGuards - plain, honestPooled, mistaken members, N 32 (one-sided sign-flip, reported, not corrected)',
        H18_9='colonyGuards - plain, honestPooled, colluding liars, N 32 (as H18_8; a two-sender quorum is expected to fail)',
        tax='plain - colonyGuards pooledShare, clean, N 32 (estimate and interval)',
        trend='per seed, the slope over log2 N (8, 16, 32) of the guard advantage with independent liars, Stage 1 rows for 8 '
              'and 16; mean, interval and two-sided sign-flip p, descriptive',
        mechanism='contamination, contests and confirmations per false value, free-riding, misreporting, action mix per cell'),
    nboot=2000, exploratory=True)

## Analysis

In [ ]:
#| export
def _test(dp, nboot, tail='greater'):
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    return dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(list(dp.values()), tail))


def stage32Summaries(rows32, rows1=(), plan=COLONY32_PLAN, nboot=2000):
    "COLONY32_PLAN: H18-7 (primary), H18-8, H18-9, the clean tax at 32, the guard trend over 8-32 (with Stage 1 rows), per-cell means."
    adv = lambda f: pairs(rows32, 'colonyGuards', 'honestPooled', f, sizes=[N32])
    hyp = dict(H18_7=_test(adv('independent'), nboot), H18_8=_test(adv('error'), nboot), H18_9=_test(adv('collude'), nboot))
    hyp['H18_7']['supported'] = bool(hyp['H18_7']['p'] < plan['primary']['alpha'] and hyp['H18_7']['estimate'] > 0)
    tax = {k: -v for k, v in pairs(rows32, 'colonyGuards', 'pooledShare', 'clean', sizes=[N32]).items()}
    sl = trendSlopes(list(rows1) + list(rows32), 'colonyGuards', [8, 16, 32])
    p2 = min(1.0, 2 * min(signFlipExact(list(sl.values()), 'greater'), signFlipExact(list(sl.values()), 'less'))) if sl else NAN
    keys = [k for k in ('honestPooled', 'pooledShare', 'contamination', 'contestFalse', 'confirmFalse', 'freeRiding', 'misreport',
                        'shareShare') if rows32 and k in rows32[0]]
    cells = {f'{a}|{f}': {k: float(np.nanmean(list(cellVals(rows32, a, f, k, [N32]).values()) or [NAN])) for k in keys}
             for a in STAGE32_ARMS for f in STAGE32_FAULTS}
    return dict(hyp=hyp, tax=dict(zip(('estimate', 'lo', 'hi'), bootSeeds(tax, nboot=nboot))),
                trend=dict(slopes=sl, estimate=float(np.mean(list(sl.values()))) if sl else NAN, p=p2), cells=cells)


def _f(x):
    return 'n/a' if x is None or not np.isfinite(x) else f'{x:.3f}'


def writeStage32(path, S, meta):
    "report.md for the 32-agent deviation."
    L = ['# Phase 18: guards at 32 agents (a user-approved deviation; exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| test | n | guards - plain | 95% interval | p (one-sided, exact) |', '|---|---|---|---|---|']
    for k, lab in (('H18_7', 'H18-7 independent liars (primary)'), ('H18_8', 'H18-8 mistaken members'), ('H18_9', 'H18-9 colluding liars')):
        t = S['hyp'][k]
        L.append(f"| {lab} | {t['n']} | {_f(t['estimate'])} | [{_f(t['ci'][0])}, {_f(t['ci'][1])}] | {_f(t['p'])} |")
    t, tr = S['tax'], S['trend']
    L += ['', f"Clean tax (plain - guards, pooled share): {_f(t['estimate'])} [{_f(t['lo'])}, {_f(t['hi'])}]",
          f"Guard advantage trend over log2 N (8, 16, 32): {_f(tr['estimate'])} per doubling, two-sided p {_f(tr['p'])}", '',
          '| cell | ' + ' | '.join(next(iter(S['cells'].values()))) + ' |', '|---|' + '---|' * len(next(iter(S['cells'].values())))]
    L += [f'| {c} | ' + ' | '.join(_f(v) for v in d.values()) + ' |' for c, d in S['cells'].items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## Tests

In [ ]:
import json, tempfile, os
from explRL.colony32 import *
from explRL.colony_grid import syntheticRows18
from explRL.coop_grid import coopHash
from explRL.llm_runs import LLMConfig

g = stage32Grid()
assert len(g) == 2 * 4 * 8 and {c['N'] for c in g} == {32}
for c in g:
    cfg = LLMConfig.fromDict(c['cfg'])
    arm, fault = c['level'].split('|')
    assert cfg.nLiar == (0 if fault == 'clean' else 8) and cfg.world.get('guards', 0) == (8 if arm == 'colonyGuards' else 0)
    assert cfg.world.get('fault', 'independent') == {'error': 'error', 'collude': 'collude'}.get(fault, 'independent')
from explRL.colony_grid import colonyEstimate
assert colonyEstimate(g)['newCalls'] == 8 * (2 * 32 * 40 + 6 * 24 * 40)          # 66,560 calls
# analysis on synthetic rows: guards +0.08 at every size (independent and error rows), no trend
rows = [dict(r, level=r['level'].replace('colony|', 'colonyGuards|')) for r in syntheticRows18(
    {'plain': (0, 0, 0, 0), 'colonyGuards': (0.08, -0.02, -0.1, 0)}, sizes=(8, 16, 32), noise=0.0)]
rows += [dict(r, level=r['level'].replace('|error', '|collude')) for r in rows if r['level'].endswith('|error')]
r1, r32 = [r for r in rows if r['N'] != 32], [r for r in rows if r['N'] == 32]
S = stage32Summaries(r32, r1, nboot=100)
assert abs(S['hyp']['H18_7']['estimate'] - 0.08) < 1e-9 and S['hyp']['H18_7']['p'] == 1 / 256 and S['hyp']['H18_7']['supported']
assert abs(S['trend']['estimate']) < 1e-9 and abs(S['tax']['estimate'] - 0.02) < 1e-9
with tempfile.TemporaryDirectory() as d:
    writeStage32(f'{d}/r.md', S, dict(mode='test'))
    assert 'H18-7' in open(f'{d}/r.md').read()
json.dumps(COLONY32_PLAN); assert coopHash(COLONY32_PLAN)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()